In [ ]:
import sys
sys.path.append("..")
import tensorflow as tf
from src.dataset import load_annotations, create_tf_dataset
from src.models import build_mobilenet_model
from src.config import MODEL_DIR
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

In [ ]:
train_df = load_annotations("annotations_train.json")
val_df = load_annotations("annotations_val.json")

train_ds = create_tf_dataset(train_df, is_training=True)
val_ds = create_tf_dataset(val_df, is_training=False)

In [ ]:
weights = compute_class_weight(
    "balanced",
    classes=np.array([0, 1, 2]), 
    y=train_df["label_id"]
    )
class_weights = dict(enumerate(weights))
print("Class weights:", class_weights)

In [ ]:
model = build_mobilenet_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(MODEL_DIR / "best_mobilenet.keras", save_best_only=True)
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# Unfreeze base model
base_model = model.get_layer("mobilenetv2_1.00_224")  # Keras base layer name
base_model.trainable = True

# Freeze bottom layers, leave top 30 trainable
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Fine-tune training
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)